# Phase 1 - Data Validation

Validates the two things Phase 1 depends on before any training starts:
1. `EndoSLAMStomachDataset` correctly indexes the real EndoSLAM folder layout (confirmed against the Kaggle mirror -- see `src/data/endoslam_dataset.py`'s module docstring).
2. `dark_degradation.py`'s synthetic dark-frame model produces something that looks like real endoscope footage, not just a dimmed photo.

**Run this on Kaggle** (dataset access; matches `config.yaml`'s data root once resolved). GPU is off -- this notebook only validates the loader, no training. Colab works too if Kaggle quota runs out -- just change the `!git clone` cell's working directory assumptions and mount the dataset from wherever you've put it; nothing else in this notebook is Kaggle-specific.

**Round-trip rule:** any further fix to `_index_sequences()` or pose/depth handling must be copied back into local `src/data/endoslam_dataset.py` and committed -- don't leave the real fix stranded in this notebook.

## 0. Setup

Clones the project repo so `src/` here matches what's tracked in git.

In [ ]:
REPO_URL = "https://github.com/ritiksharma3/endoslam.git"
assert REPO_URL, "Set REPO_URL to the pushed GitHub repo before running on Kaggle"

!git clone $REPO_URL repo
%cd repo
!pip install -q -r environment/requirements.txt  # does NOT include torch -- Kaggle ships it preinstalled

import sys
sys.path.insert(0, ".")

## 1. Resolve the dataset mount

Real layout was confirmed in earlier runs (see `src/data/endoslam_dataset.py`'s
module docstring) -- real cameras nest under `Cameras/{cam}/Stomach-*/TumorfreeTrajectory_*/`,
UnityCam sits at the top level as a single flat sequence. The one thing that
still varies across Kaggle environments is where `/kaggle/input` actually
mounts the dataset, so resolve that first.

In [ ]:
import os

def find_endoslam_root(base="/kaggle/input", max_depth=4):
    """Newer Kaggle kernels sometimes nest dataset mounts (e.g. under an
    extra 'datasets/' layer) instead of /kaggle/input/<slug> directly --
    search a few levels down for the actual EndoSLAM root instead of
    assuming a fixed path."""
    for root, dirs, _files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        if os.path.basename(root).lower() == "endoslam":
            return root
    return None

print("mounted under /kaggle/input:", os.listdir("/kaggle/input"))

DATA_ROOT = find_endoslam_root()
assert DATA_ROOT, "could not find an 'endoslam' dir anywhere under /kaggle/input within depth 4"
print(f"DATA_ROOT resolved to: {DATA_ROOT}")
print("top level:", sorted(os.listdir(DATA_ROOT)))

## 2. Dataset loader smoke test

In [ ]:
import yaml

with open("configs/config.yaml") as f:
    config = yaml.safe_load(f)

config["data"]["root"] = DATA_ROOT
config

In [ ]:
from src.data.endoslam_dataset import EndoSLAMStomachDataset

all_cameras = [config["data"]["synthetic_cam"]] + config["data"]["real_cams"]

train_ds = EndoSLAMStomachDataset(
    config, split="train", cameras=all_cameras,
    context_window=config["reconstruction"]["context_window"],
)
val_ds = EndoSLAMStomachDataset(
    config, split="val", cameras=all_cameras,
    context_window=config["reconstruction"]["context_window"],
)
test_ds = EndoSLAMStomachDataset(
    config, split="test", cameras=all_cameras,
    context_window=config["reconstruction"]["context_window"],
)

print(f"train windows: {len(train_ds)}")
print(f"val windows:   {len(val_ds)}")
print(f"test windows:  {len(test_ds)}")
assert len(train_ds) > 0, "empty dataset -- _index_sequences() almost certainly needs patching, see CHECKPOINT above"

In [ ]:
from src.data.endoslam_dataset import EndoSLAMStomachDataset

all_cameras = [config["data"]["synthetic_cam"]] + config["data"]["real_cams"]

train_ds = EndoSLAMStomachDataset(
    config, split="train", cameras=all_cameras,
    context_window=config["reconstruction"]["context_window"],
)
val_ds = EndoSLAMStomachDataset(
    config, split="val", cameras=all_cameras,
    context_window=config["reconstruction"]["context_window"],
)
test_ds = EndoSLAMStomachDataset(
    config, split="test", cameras=all_cameras,
    context_window=config["reconstruction"]["context_window"],
)

print(f"train windows: {len(train_ds)}")
print(f"val windows:   {len(val_ds)}")
print(f"test windows:  {len(test_ds)}")
assert len(train_ds) > 0, "empty dataset -- _index_sequences() needs further patching"

## 3. Dark-degradation visual check on real frames

The module's own `__main__` self-test only checks a synthetic gradient. This checks it on an actual EndoSLAM frame -- it should look like dim, slightly blurred endoscope footage, not just a darkened photo.

In [ ]:
import matplotlib.pyplot as plt
from src.data.dark_degradation import build_paired_dataset_entry

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col in range(4):
    sample = train_ds[col * max(1, len(train_ds) // 4)]
    clean = sample["images"][0].permute(1, 2, 0).numpy()  # first frame of the window, CHW -> HWC
    pair = build_paired_dataset_entry(clean, config)

    axes[0, col].imshow(pair["clean"])
    axes[0, col].set_title(f"clean ({sample['camera']})")
    axes[0, col].axis("off")
    axes[1, col].imshow(pair["dark"])
    axes[1, col].set_title("degraded")
    axes[1, col].axis("off")
plt.tight_layout()
plt.show()

## Done

If the smoke test passed and the degraded frames look plausibly dark/blurred (not just dimmed), Phase 1 is validated. Copy any `_index_sequences()` / `_load_poses()` patches back to local `src/`, commit, and update `PROGRESS.md` before starting Phase 2.